### **영화 리뷰의 감성 판별하기**
단어 임베딩 + Dense레이어로 리뷰가 긍정적인지, 부정적인지 확인하는 예제로 보인다  
202435119 한여원

In [2]:
import numpy as np

import tensorflow as tf
from tensorflow import keras

import matplotlib.pyplot as plt

In [13]:
imdb = keras.datasets.imdb

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)


d:\hanyeowon\Coding\pythonworks\deep\.venv\Lib\site-packages\numpy\lib\_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


In [14]:
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

# 정수 인코딩 되어있음을 확인하자
print(x_train[0])

(25000,)
(25000,)
(25000,)
(25000,)
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]


In [15]:
np.unique(y_train, return_counts=True)

(array([0, 1]), array([12500, 12500]))

In [16]:
word_to_index = imdb.get_word_index()

word_to_index = {k:(v+3) for k,v in word_to_index.items()}
word_to_index["<PAD>"] = 0
word_to_index["<START>"] = 1
word_to_index["<UNK>"] = 2
word_to_index["<UNUSED>"] = 3

index_to_word = dict([(value, key) for (key, value) in word_to_index.items()])

In [17]:
print(' '.join([index_to_word[index] for index in x_train[0]]))

<START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for <UNK> and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also <UNK> to the two little boy's that played the <UNK> of norman and paul they were just brilliant children are often left out of the <UNK> list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for wha

In [18]:
# 단어들을 임베딩한다.
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import *

x_train = pad_sequences(x_train, maxlen=100)
x_test = pad_sequences(x_test, maxlen=100)

In [19]:
print(len(x_train[0]), len(x_train[1]))
print(x_train[0])


100 100
[1415   33    6   22   12  215   28   77   52    5   14  407   16   82
    2    8    4  107  117 5952   15  256    4    2    7 3766    5  723
   36   71   43  530  476   26  400  317   46    7    4    2 1029   13
  104   88    4  381   15  297   98   32 2071   56   26  141    6  194
 7486   18    4  226   22   21  134  476   26  480    5  144   30 5535
   18   51   36   28  224   92   25  104    4  226   65   16   38 1334
   88   12   16  283    5   16 4472  113  103   32   15   16 5345   19
  178   32]


In [24]:
vocab_size = 10000

model = Sequential()
model.add(Input(shape=(100,)))
model.add(Embedding(vocab_size, 64))

model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 6400)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       409,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,049,729 (4.00 MB)

 Trainable params: 1,049,729 (4.00 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

history = model.fit(x_train, y_train, batch_size=64, epochs=20, verbose=1, validation_data=(x_test, y_test))

Epoch 1/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.7613 - loss: 0.4679 - val_accuracy: 0.8274 - val_loss: 0.3775
Epoch 2/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9378 - loss: 0.1749 - val_accuracy: 0.8329 - val_loss: 0.4002
Epoch 3/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9937 - loss: 0.0287 - val_accuracy: 0.8328 - val_loss: 0.5256
Epoch 4/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9989 - loss: 0.0064 - val_accuracy: 0.8342 - val_loss: 0.6056
Epoch 5/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9996 - loss: 0.0027 - val_accuracy: 0.8351 - val_loss: 0.6569
Epoch 6/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 1.0000 - loss: 8.3281e-04 - val_accuracy: 0.8366 - val_loss: 0.6914
Epoch 7/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 1.0000 - loss: 5.8178e-04 - val_accuracy: 0.8372 - val_loss: 0.7215
Epoch 8/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9999 - loss: 6.9670e-04 - val_

In [28]:
results = model.evaluate(x_test, y_test, verbose=2)
print(results)

782/782 - 1s - 1ms/step - accuracy: 0.8227 - loss: 1.2828
[1.2827951908111572, 0.8226799964904785]


In [37]:
review1 = "Back to the Future is an absolute masterpiece that never gets old. Even after all these years, the story of Marty McFly and Doc Brown traveling to 1955 remains incredibly entertaining. The plot is perfectly written, with every small detail in the beginning paying off beautifully by the end. The chemistry between Michael J. Fox and Christopher Lloyd is unmatched, and the DeLorean is arguably the coolest time machine in cinema history. It’s a flawless blend of sci-fi, comedy, and heart. If you somehow haven't seen this timeless classic yet, do yourself a favor and watch it immediately."
review2 = "If you are looking for a pure, feel-good adventure, this is the ultimate choice. The concept of accidentally interfering with your own parents' love story is both hilarious and surprisingly grounded. The characters are incredibly memorable, especially the cowardly yet lovable George McFly and the bully Biff Tannen. While the sci-fi elements are soft and rely heavily on cinematic logic, the movie compensates with witty dialogue and fantastic comedic timing. It is a masterclass in blockbuster entertainment. It does not try to be overly dark or complex, just incredibly fun."
review3 = "While I understand why people consider Back to the Future a classic, viewing it today reveals some highly uncomfortable elements. The romantic subplot involving Marty and his teenage mother is deeply weird and ages poorly, creating a genuinely cringeworthy atmosphere during the middle act. Additionally, the plot relies on far too many convenient coincidences to resolve its conflicts. The pacing in the first twenty minutes feels sluggish before the time travel actually occurs. It definitely has historical value in cinema, but the charm did not completely work for me. It feels very dated."
review4 = "Honestly, I found this highly overrated film to be quite disappointing and childish. The science fiction logic makes absolutely no sense, and the rules of time travel change whenever it suits the predictable plot. Biff is a cartoonish, one-dimensional villain whose constant bullying becomes annoying rather than threatening. Marty is an obnoxious protagonist who seems to succeed mostly through pure luck rather than intelligence. The special effects look incredibly cheap by modern standards, and the ending feels like a lazy setup for a sequel. I struggled to stay awake during this boring story."
reviews = [review1, review2, review3, review4]

import re
for rev in reviews:
    rev = re.sub("[^0-9a-zA-Z ]", "", rev).lower()
    print(rev)

back to the future is an absolute masterpiece that never gets old even after all these years the story of marty mcfly and doc brown traveling to 1955 remains incredibly entertaining the plot is perfectly written with every small detail in the beginning paying off beautifully by the end the chemistry between michael j fox and christopher lloyd is unmatched and the delorean is arguably the coolest time machine in cinema history its a flawless blend of scifi comedy and heart if you somehow havent seen this timeless classic yet do yourself a favor and watch it immediately
if you are looking for a pure feelgood adventure this is the ultimate choice the concept of accidentally interfering with your own parents love story is both hilarious and surprisingly grounded the characters are incredibly memorable especially the cowardly yet lovable george mcfly and the bully biff tannen while the scifi elements are soft and rely heavily on cinematic logic the movie compensates with witty dialogue and 

In [ ]:
for review in reviews:
    review_encoding = []
    for w in review.split():
        index = word_to_index.get(w, 2)
        if index <= 10000:
            review_encoding.append(index)
        else:
            review_encoding.append(word_to_index["<UNK>"])

    test_input = pad_sequences([review_encoding], maxlen=100)
    value = model.predict(test_input)
    print(value)
    if (value > 0.5):
        print("긍정적")
    else:
        print("부정적")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
[[1.]]
긍정적
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
[[0.9996161]]
긍정적
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
[[0.25253952]]
부정적
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
[[3.200813e-06]]
부정적


### **LSTM을 써보면 안되는 것일까? 한번 해보도록 하자**

In [44]:
# LSTM을 집어넣은 새 모델 구축
model = Sequential()
model.add(Input(shape=(100,)))
model.add(Embedding(vocab_size, 64))

model.add(LSTM(64))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

In [45]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

history = model.fit(x_train, y_train, batch_size=64, epochs=20, verbose=1, validation_data=(x_test, y_test))

Epoch 1/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 37s 90ms/step - accuracy: 0.7874 - loss: 0.4380 - val_accuracy: 0.8448 - val_loss: 0.3569
Epoch 2/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 24s 62ms/step - accuracy: 0.8879 - loss: 0.2785 - val_accuracy: 0.8474 - val_loss: 0.3463
Epoch 3/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 25s 64ms/step - accuracy: 0.9153 - loss: 0.2187 - val_accuracy: 0.8401 - val_loss: 0.3811
Epoch 4/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 28s 72ms/step - accuracy: 0.9366 - loss: 0.1664 - val_accuracy: 0.8396 - val_loss: 0.4506
Epoch 5/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 31s 79ms/step - accuracy: 0.9556 - loss: 0.1218 - val_accuracy: 0.8342 - val_loss: 0.5237
Epoch 6/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 20s 51ms/step - accuracy: 0.9665 - loss: 0.0951 - val_accuracy: 0.8316 - val_loss: 0.6303
Epoch 7/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 19s 48ms/step - accuracy: 0.9774 - loss: 0.0656 - val_accuracy: 0.8277 - val_loss: 0.7429
Epoch 8/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 19s 49ms/step - accuracy: 0.9820 - loss: 0.0547 - 

In [46]:
for review in reviews:
    review_encoding = []
    for w in review.split():
        index = word_to_index.get(w, 2)
        if index <= 10000:
            review_encoding.append(index)
        else:
            review_encoding.append(word_to_index["<UNK>"])

    test_input = pad_sequences([review_encoding], maxlen=100)
    value = model.predict(test_input)
    print(value)
    if (value > 0.5):
        print("긍정적")
    else:
        print("부정적")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step
[[0.99984896]]
긍정적
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
[[0.9990521]]
긍정적
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
[[0.00386397]]
부정적
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
[[1.4570578e-07]]
부정적


##### **일단 확실히 부정적인 리뷰들을 더 부정적으로 보는 거 같기는 하다**